# Dogs vs Cats Training Notebook
Notebook này tương đương logic với `main.py` trong thư mục `dogsandcat`.

In [ ]:
# Colab: install KaggleHub (safe to run multiple times)
%pip -q install kagglehub

In [ ]:
import os
import sys
import importlib
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, precision_recall_fscore_support

KAGGLE_DATASET_HANDLE = 'bhavikjikadara/dog-and-cat-classification-dataset'
IMG_SIZE = (96, 96)
BATCH_SIZE = 16
EPOCHS = 10
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15
SEED = 42

IN_COLAB = 'google.colab' in sys.modules
BASE_DIR = '/content' if IN_COLAB else os.getcwd()
DATASET_DIR = os.path.join(BASE_DIR, 'dataset')
TRAIN_DIR = os.path.join(DATASET_DIR, 'train')
TEST_DIR = os.path.join(DATASET_DIR, 'test')


def has_image_files(directory_path: Path) -> bool:
    image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp'}
    for root, _, files in os.walk(directory_path):
        for file_name in files:
            if Path(file_name).suffix.lower() in image_exts:
                return True
    return False


def find_class_root(start_dir: Path) -> Path | None:
    for root, dirs, _ in os.walk(start_dir):
        if len(dirs) < 2:
            continue
        root_path = Path(root)
        class_dirs = [root_path / d for d in dirs]
        if sum(has_image_files(d) for d in class_dirs) >= 2:
            return root_path
    return None


def resolve_data_dirs() -> tuple[str, str | None, str | None]:
    if os.path.isdir(DATASET_DIR):
        train_dir = TRAIN_DIR if os.path.isdir(TRAIN_DIR) else None
        test_dir = TEST_DIR if os.path.isdir(TEST_DIR) else None
        return DATASET_DIR, train_dir, test_dir

    kagglehub = importlib.import_module('kagglehub')
    print(f'Tai dataset tu Kaggle: {KAGGLE_DATASET_HANDLE}')
    downloaded_path = Path(kagglehub.dataset_download(KAGGLE_DATASET_HANDLE))
    print(f'Path dataset tu KaggleHub: {downloaded_path}')

    class_root = find_class_root(downloaded_path)
    if class_root is None:
        raise FileNotFoundError(f'Khong tim duoc class root trong: {downloaded_path}')

    train_candidate = None
    test_candidate = None
    for child in class_root.iterdir():
        if not child.is_dir():
            continue
        child_name = child.name.lower()
        if 'train' in child_name:
            train_candidate = str(child)
        elif 'test' in child_name:
            test_candidate = str(child)

    if train_candidate and test_candidate:
        return str(class_root), train_candidate, test_candidate

    return str(class_root), None, None


gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'Su dung GPU: {gpus}')
else:
    print('Khong co GPU CUDA kha dung, se train bang CPU.')

resolved_dataset_dir, resolved_train_dir, resolved_test_dir = resolve_data_dirs()
print(f'Dataset root dang dung: {resolved_dataset_dir}')

In [ ]:
has_train_test_layout = resolved_train_dir is not None and resolved_test_dir is not None

if has_train_test_layout:
    train_ds, val_ds = tf.keras.utils.image_dataset_from_directory(
        resolved_train_dir,
        labels='inferred',
        label_mode='categorical',
        color_mode='rgb',
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        validation_split=VAL_SPLIT,
        subset='both',
        seed=SEED,
        shuffle=True,
    )

    test_ds = tf.keras.utils.image_dataset_from_directory(
        resolved_test_dir,
        labels='inferred',
        label_mode='categorical',
        color_mode='rgb',
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    class_names = train_ds.class_names

    if class_names != test_ds.class_names:
        raise ValueError(
            'Class train/test khong khop. ' +
            f'Train: {class_names}, Test: {test_ds.class_names}'
        )
else:
    combined_val_test_split = VAL_SPLIT + TEST_SPLIT
    train_ds, val_test_ds = tf.keras.utils.image_dataset_from_directory(
        resolved_dataset_dir,
        labels='inferred',
        label_mode='categorical',
        color_mode='rgb',
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        validation_split=combined_val_test_split,
        subset='both',
        seed=SEED,
        shuffle=True,
    )

    val_test_batches = tf.data.experimental.cardinality(val_test_ds).numpy()
    if val_test_batches <= 1:
        raise ValueError(
            'Tap du lieu qua it de tach rieng validation va test. ' +
            'Hay bo sung du lieu hoac giam BATCH_SIZE.'
        )

    test_batches = max(1, int(round(val_test_batches * TEST_SPLIT / combined_val_test_split)))
    if test_batches >= val_test_batches:
        test_batches = val_test_batches - 1

    test_ds = val_test_ds.take(test_batches)
    val_ds = val_test_ds.skip(test_batches)
    class_names = train_ds.class_names

num_classes = len(class_names)

train_ds = train_ds.apply(tf.data.experimental.ignore_errors())
val_ds = val_ds.apply(tf.data.experimental.ignore_errors())
test_ds = test_ds.apply(tf.data.experimental.ignore_errors())

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

print(f'Number of classes: {num_classes}')
print(f'Class names: {class_names}')

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

model = keras.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    layers.Rescaling(1.0 / 255),
    data_augmentation,
    layers.Conv2D(8, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax'),
])

model.summary()

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
    )
]

history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=callbacks,
)

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
print('\n--- ĐÁNH GIÁ TRÊN TẬP VALIDATION ---')
val_loss, val_accuracy = model.evaluate(val_ds)
print(f'Validation Loss: {val_loss:.4f}')
print(f'Validation Accuracy: {val_accuracy:.4f}')

print('\n--- ĐÁNH GIÁ TRÊN TẬP TEST ---')
test_loss, test_accuracy = model.evaluate(test_ds)
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy:.4f}')

y_pred = model.predict(test_ds)
y_pred_classes = np.argmax(y_pred, axis=1)

y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_ds], axis=0)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred_classes,
    average='weighted',
    zero_division=0,
)

print('\n--- TỔNG HỢP CHỈ SỐ TEST ---')
print(f'Accuracy : {test_accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall   : {recall:.4f}')
print(f'F1-score : {f1:.4f}')

print('\n--- CLASSIFICATION REPORT ---')
print(classification_report(y_true, y_pred_classes, target_names=class_names, zero_division=0))

final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
acc_gap = final_train_acc - final_val_acc

print('\n--- PHÂN TÍCH NHANH ---')
if acc_gap > 0.1:
    print(
        'Mô hình có dấu hiệu overfitting: train_acc cao hơn val_acc đáng kể. ' +
        'Có thể tăng augmentation hoặc regularization.'
    )
elif final_train_acc < 0.7 and final_val_acc < 0.7:
    print(
        'Mô hình có dấu hiệu underfitting: cả train_acc và val_acc đều thấp. ' +
        'Có thể tăng số epoch hoặc mở rộng mô hình vừa phải.'
    )
else:
    print('Mô hình đang fit tương đối ổn, chưa thấy dấu hiệu overfit/underfit rõ rệt.')

In [ ]:
# Export metrics + result images for upload form
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

if 'history' not in globals() or 'y_true' not in globals() or 'y_pred_classes' not in globals():
    raise RuntimeError('Hay chay cac cell train/evaluate truoc khi export ket qua.')

export_dir = os.path.join(BASE_DIR, 'submission_assets')
os.makedirs(export_dir, exist_ok=True)

# 1) Metrics for form fields
final_metrics = {
    'accuracy_r2': float(test_accuracy),
    'loss_mse': float(test_loss),
    'precision': float(precision),
    'recall': float(recall),
    'f1_score': float(f1),
    'num_classes': int(len(class_names)),
    'class_names': list(class_names),
}

with open(os.path.join(export_dir, 'metrics.json'), 'w', encoding='utf-8') as f:
    json.dump(final_metrics, f, ensure_ascii=False, indent=2)

with open(os.path.join(export_dir, 'upload_text.txt'), 'w', encoding='utf-8') as f:
    f.write(f"Accuracy / R2 Score: {final_metrics['accuracy_r2']:.4f}\n")
    f.write(f"Loss / MSE: {final_metrics['loss_mse']:.4f}\n")
    f.write(
        'Other Metrics: '
        f"Precision={final_metrics['precision']:.4f}, "
        f"Recall={final_metrics['recall']:.4f}, "
        f"F1={final_metrics['f1_score']:.4f}\n"
    )

# 2) Image 1: Training curves
fig = plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.title('Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
curve_path = os.path.join(export_dir, '01_training_curves.png')
fig.savefig(curve_path, dpi=150, bbox_inches='tight')
plt.close(fig)

# 3) Image 2: Confusion matrix
cm = confusion_matrix(y_true, y_pred_classes)
fig = plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation='nearest', cmap='Blues')
plt.title('Confusion Matrix')
plt.colorbar()
tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names, rotation=45)
plt.yticks(tick_marks, class_names)
plt.xlabel('Predicted')
plt.ylabel('True')

thresh = cm.max() / 2.0 if cm.size > 0 else 0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(
            j,
            i,
            str(cm[i, j]),
            ha='center',
            va='center',
            color='white' if cm[i, j] > thresh else 'black',
        )

plt.tight_layout()
cm_path = os.path.join(export_dir, '02_confusion_matrix.png')
fig.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.close(fig)

# 4) Image 3: Sample predictions
sample_images = None
sample_labels = None
for batch_images, batch_labels in test_ds.take(1):
    sample_images = batch_images.numpy()
    sample_labels = np.argmax(batch_labels.numpy(), axis=1)

if sample_images is not None:
    sample_preds = model.predict(sample_images, verbose=0)
    sample_pred_classes = np.argmax(sample_preds, axis=1)

    n_show = min(9, len(sample_images))
    fig = plt.figure(figsize=(9, 9))
    for i in range(n_show):
        plt.subplot(3, 3, i + 1)
        plt.imshow(sample_images[i].astype('uint8'))
        true_name = class_names[sample_labels[i]]
        pred_name = class_names[sample_pred_classes[i]]
        color = 'green' if sample_labels[i] == sample_pred_classes[i] else 'red'
        plt.title(f'T:{true_name} | P:{pred_name}', color=color, fontsize=9)
        plt.axis('off')

    plt.tight_layout()
    pred_path = os.path.join(export_dir, '03_sample_predictions.png')
    fig.savefig(pred_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

print('=== THONG TIN NHAP FORM ===')
print(f"Accuracy / R2 Score: {final_metrics['accuracy_r2']:.4f}")
print(f"Loss / MSE: {final_metrics['loss_mse']:.4f}")
print(
    'Other Metrics: '
    f"Precision={final_metrics['precision']:.4f}, "
    f"Recall={final_metrics['recall']:.4f}, "
    f"F1={final_metrics['f1_score']:.4f}"
)

print('\n=== FILE SAN SANG DE UPLOAD ===')
for file_name in sorted(os.listdir(export_dir)):
    print('-', os.path.join(export_dir, file_name))

print('\nGoi y upload toi da 5 anh:')
print('- 01_training_curves.png')
print('- 02_confusion_matrix.png')
print('- 03_sample_predictions.png')